In [ ]:
import numpy as np
Q = np.array([[2, 1],
              [1, 3]])
c = np.array([1, -2])
a = np.array([1, 1])
b = 2

# ----- Giải bằng Lagrange - KKT-----
# [Q  a][x] = [-c]
# [a^T 0][λ]   [ b]

KKT = np.block([
    [Q, a.reshape(-1,1)],
    [a.reshape(1,-1), np.zeros((1,1))]
])

rhs = np.concatenate([-c, [b]])

sol = np.linalg.solve(KKT, rhs)

x_star = sol[:2]
lam = sol[2]

# Giá trị tối ưu
f_star = 0.5 * x_star.T @ Q @ x_star + c.T @ x_star

print("===== CÂU 1 =====")
print("x* =", x_star)
print("lambda =", lam)
print("f(x*) =", f_star)


In [ ]:
import sympy as sp

x1 = sp.symbols('x1', real=True)
x2 = 2 - x1

x = sp.Matrix([x1, x2])
Q_sym = sp.Matrix(Q)
c_sym = sp.Matrix(c)

# Hàm mục tiêu sau khi khử ràng buộc
f = 0.5 * (x.T * Q_sym * x)[0] + (c_sym.T * x)[0]

# Đạo hàm và giải
df = sp.diff(f, x1)
x1_star = sp.solve(df, x1)[0]
x2_star = 2 - x1_star

f_star = f.subs(x1, x1_star)

print("=== CÁCH 2: KHỬ RÀNG BUỘC ===")
print("x* =", [float(x1_star), float(x2_star)])
print("f(x*) =", float(f_star))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----- Dữ liệu Câu 2 -----
P = np.array([[3, 0],
              [0, 2]])
q = np.array([3, -4])

def f(x):
    return 0.5 * x.T @ P @ x + q.T @ x + 2

def grad(x):
    return P @ x + q

# ----- Nghiệm tối ưu -----
x_star = -np.linalg.inv(P) @ q
p_star = f(x_star)

print("\n===== CÂU 2a =====")
print("x* =", x_star)
print("p* =", p_star)

# ----- CÂU 2b: Least Squares -----
A = np.diag([np.sqrt(3), np.sqrt(2)])
b_ls = -np.linalg.inv(A.T) @ q

print("\n===== CÂU 2b =====")
print("A =\n", A)
print("b =", b_ls)

# ----- Gradient Descent -----
def gradient_descent(alpha, x0, max_iter=100):
    x = x0.copy()
    xs, fs = [], []

    for k in range(max_iter):
        xs.append(x.copy())
        fs.append(f(x))
        print(f"Iter {k:3d}: x = {x}, f(x) = {f(x):.6f}")
        x = x - alpha * grad(x)

    return np.array(xs), np.array(fs)

x0 = np.array([1.0, -2.0])

print("\n===== CÂU 2c – lr = 0.2 =====")
xs_02, fs_02 = gradient_descent(0.2, x0)

print("\n===== CÂU 2c – lr = 0.6 =====")
xs_06, fs_06 = gradient_descent(0.6, x0)

# ----- Đồ thị sai số -----
plt.figure()
plt.plot(np.abs(fs_02 - p_star), label="lr = 0.2")
plt.plot(np.abs(fs_06 - p_star), label="lr = 0.6")
plt.xlabel("Iteration")
plt.ylabel("|f(xk) - p*|")
plt.title("CÂU 2c – Error convergence")
plt.legend()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

P = np.array([[3, 0],
              [0, 2]], dtype=float)

q = np.array([3, -4], dtype=float)
def f(x):
    return 0.5 * x.T @ P @ x + q.T @ x + 2

def grad_f(x):
    return P @ x + q
x_star = -np.linalg.solve(P, q)
p_star = f(x_star)

print("x* =", x_star)
print("p* =", p_star)
def gradient_descent(x0, lr, max_iter=100):
    x = x0.copy()
    xs = []
    fs = []

    for k in range(1, max_iter + 1):
        x = x - lr * grad_f(x)
        xs.append(x.copy())
        fs.append(f(x))

        if k % 10 == 0:
            print(f"lr={lr}, iter={k}, x={x}, f(x)={f(x)}")

    return np.array(xs), np.array(fs)
x0 = np.array([1, -2], dtype=float)

xs_02, fs_02 = gradient_descent(x0, lr=0.2)
xs_06, fs_06 = gradient_descent(x0, lr=0.6)
plt.figure()
plt.plot(np.abs(fs_02 - p_star), label="lr = 0.2")
plt.plot(np.abs(fs_06 - p_star), label="lr = 0.6")
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("|f(x(k)) - p*|")
plt.legend()
plt.grid()
plt.show()


In [ ]:
x1 = np.linspace(-3, 2, 200)
x2 = np.linspace(-3, 3, 200)
X1, X2 = np.meshgrid(x1, x2)

Z = np.zeros_like(X1)
for i in range(X1.shape[0]):
    for j in range(X1.shape[1]):
        Z[i, j] = f(np.array([X1[i, j], X2[i, j]]))

plt.figure()
plt.contour(X1, X2, Z, levels=30)
plt.plot(xs_02[:,0], xs_02[:,1], 'o-', label="lr=0.2")
plt.plot(xs_06[:,0], xs_06[:,1], 'x--', label="lr=0.6")
plt.plot(x_star[0], x_star[1], 'r*', markersize=12, label="x*")
plt.legend()
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Gradient Descent Trajectories")
plt.grid()
plt.show()
